# ❄️ NYC Snow Map — Data Pipeline### Winter 2025–2026This notebook builds all data files needed for the interactive NYC neighborhood-level snowfall visualization.**Outputs:**- `nyc_neighborhoods.geojson` — ~200 NTAs with snowfall properties- `nyc_boroughs.geojson` — 5 boroughs with aggregated totals- `nyc_storms.json` — individual storm events- `nyc_historical.json` — historical season totals (Central Park, 1869–present)**Architecture:** Neighborhood Tabulation Areas (NTAs) instead of street segments. Borough-level default view, neighborhood drill-down on zoom.

In [1]:
# CELL 1 — Install dependencies
!pip install geopandas scipy shapely requests pandas numpy CoCoRaHS-Download-Tool -q

  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [2]:
# CELL 2 — Imports
import os, json, requests, time, glob
import pandas as pd
import numpy as np
import geopandas as gpd
from scipy.spatial import cKDTree
from shapely.geometry import Point
from collections import defaultdict
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

os.makedirs('outputs', exist_ok=True)
print('Imports ready')

Imports ready


In [3]:
# CELL 3 — CONFIGURATION  ← paste your NOAA key here
NOAA_API_KEY   = 'NDJeXTCmRuSQrLzNQVPNZhvDLsMrAytO'

# Expanded bounding box: NYC + nearby NJ (Newark, Jersey City, Teterboro area)
NYC_BBOX = dict(lat_min=40.49, lat_max=40.92, lon_min=-74.30, lon_max=-73.70)

# NOAA GHCN stations — 5 stations ringing the NYC metro
NOAA_STATIONS = {
    'GHCND:USW00094728': {'name': 'Central Park',        'lat': 40.7789, 'lon': -73.9692},
    'GHCND:USW00014732': {'name': 'LaGuardia Airport',   'lat': 40.7794, 'lon': -73.8803},
    'GHCND:USW00094789': {'name': 'JFK Airport',         'lat': 40.6413, 'lon': -73.7623},
    'GHCND:USW00014734': {'name': 'Newark Airport',      'lat': 40.6832, 'lon': -74.1687},
    'GHCND:USW00094741': {'name': 'Teterboro Airport',   'lat': 40.8598, 'lon': -74.0593},
}

# Primary station for historical record (Central Park has the longest record)
PRIMARY_STATION_ID   = 'GHCND:USW00094728'
PRIMARY_STATION_NAME = 'Central Park, NY'

SEASON_START = '2025-10-01'
SEASON_END   = '2026-03-31'
IDW_POWER    = 2

# CoCoRaHS config — fetch BOTH NY and NJ for the metro area
COCORAHS_STATES = ['NY', 'NJ']
COCORAHS_START  = '2025-10-01'
COCORAHS_END    = '2026-03-31'

print('Config set — NYC metro mode (multi-station)')
print(f'NOAA stations: {len(NOAA_STATIONS)}')
for sid, info in NOAA_STATIONS.items():
    print(f'  {info["name"]:25s} {sid}  ({info["lat"]:.4f}, {info["lon"]:.4f})')
print(f'CoCoRaHS states: {COCORAHS_STATES}')
print(f'Bbox: {NYC_BBOX}')

Config set — NYC metro mode (multi-station)
NOAA stations: 5
  Central Park              GHCND:USW00094728  (40.7789, -73.9692)
  LaGuardia Airport         GHCND:USW00014732  (40.7794, -73.8803)
  JFK Airport               GHCND:USW00094789  (40.6413, -73.7623)
  Newark Airport            GHCND:USW00014734  (40.6832, -74.1687)
  Teterboro Airport         GHCND:USW00094741  (40.8598, -74.0593)
CoCoRaHS states: ['NY', 'NJ']
Bbox: {'lat_min': 40.49, 'lat_max': 40.92, 'lon_min': -74.3, 'lon_max': -73.7}


## 1. Fetch CoCoRaHS Data (NYC + NJ Metro)
Automated via `CoCoRaHS-Download-Tool` pip package — fetches both NY and NJ state data.

In [4]:
# CELL 4 — Fetch CoCoRaHS daily reports for NY + NJ via CoCoRaHS-Download-Tool
import subprocess, io

all_dfs = []

for state in COCORAHS_STATES:
    print(f'Fetching CoCoRaHS data for state={state}, {COCORAHS_START} to {COCORAHS_END}...')
    try:
        result = subprocess.run(
            ['python', '-m', 'cocorahs_dlt',
             '--state', state,
             'data',
             '--start_date', COCORAHS_START,
             '--end_date', COCORAHS_END,
             '--format', 'CSV'],
            capture_output=True, text=True, timeout=120
        )
        if result.returncode == 0 and len(result.stdout) > 100:
            df_state = pd.read_csv(io.StringIO(result.stdout))
            df_state['_source_state'] = state
            all_dfs.append(df_state)
            print(f'  {state}: {len(df_state):,} rows')
        else:
            print(f'  {state}: CLI returned no data')
            print(f'  stderr: {result.stderr[:300]}')
    except Exception as e:
        print(f'  {state}: fetch failed: {e}')

# Fallback: try web export
if not all_dfs:
    print('\nTrying direct CoCoRaHS web export as fallback...')
    for state in COCORAHS_STATES:
        try:
            url = (f'https://data.cocorahs.org/cocorahs/export/exportreports.aspx'
                   f'?ReportType=Daily&Format=CSV&State={state}'
                   f'&StartDate={COCORAHS_START}&EndDate={COCORAHS_END}')
            r = requests.get(url, timeout=60)
            if r.status_code == 200 and len(r.text) > 100:
                df_state = pd.read_csv(io.StringIO(r.text))
                df_state['_source_state'] = state
                all_dfs.append(df_state)
                print(f'  {state} web export: {len(df_state):,} rows')
        except Exception as e:
            print(f'  {state} web export failed: {e}')

if all_dfs:
    df_raw = pd.concat(all_dfs, ignore_index=True)
    print(f'\nTotal CoCoRaHS rows: {len(df_raw):,}')
else:
    df_raw = None
    print('No CoCoRaHS data — will use NOAA multi-station only')

if df_raw is not None:
    df_raw.columns = df_raw.columns.str.strip()
    df_raw = df_raw.apply(lambda c: c.str.strip() if c.dtype == 'object' else c)

    # Detect column names
    date_col = next((c for c in df_raw.columns if 'date' in c.lower() and 'obs' in c.lower()),
                    next((c for c in df_raw.columns if 'date' in c.lower()), None))
    lat_col  = next((c for c in df_raw.columns if 'lat' in c.lower()), None)
    lon_col  = next((c for c in df_raw.columns if 'lon' in c.lower()), None)
    snow_col = next((c for c in df_raw.columns if 'snow' in c.lower() and 'new' in c.lower()),
                    next((c for c in df_raw.columns if 'snow' in c.lower()), None))
    stn_col  = next((c for c in df_raw.columns if 'station' in c.lower() and 'num' in c.lower()),
                    next((c for c in df_raw.columns if 'station' in c.lower()), None))
    name_col_c = next((c for c in df_raw.columns if 'station' in c.lower() and 'name' in c.lower()), stn_col)

    print(f'Detected columns: date={date_col}, lat={lat_col}, lon={lon_col}, snow={snow_col}, station={stn_col}')

    # Rename to standard names
    rename_map = {}
    if date_col and date_col != 'ObservationDate': rename_map[date_col] = 'ObservationDate'
    if lat_col and lat_col != 'Latitude': rename_map[lat_col] = 'Latitude'
    if lon_col and lon_col != 'Longitude': rename_map[lon_col] = 'Longitude'
    if snow_col and snow_col != 'NewSnowDepth': rename_map[snow_col] = 'NewSnowDepth'
    if stn_col and stn_col != 'StationNumber': rename_map[stn_col] = 'StationNumber'
    if name_col_c and name_col_c != 'StationName': rename_map[name_col_c] = 'StationName'
    df_raw = df_raw.rename(columns=rename_map)

    df_raw['ObservationDate'] = pd.to_datetime(df_raw['ObservationDate'])
    df_raw['Latitude']        = pd.to_numeric(df_raw['Latitude'],        errors='coerce')
    df_raw['Longitude']       = pd.to_numeric(df_raw['Longitude'],       errors='coerce')
    df_raw['NewSnowDepth']    = pd.to_numeric(df_raw['NewSnowDepth'],    errors='coerce').fillna(0)

    df = df_raw[(df_raw['ObservationDate'] >= SEASON_START) &
                (df_raw['ObservationDate'] <= SEASON_END)].copy()

    # Filter to expanded NYC metro bounding box (includes NJ side)
    mask = (df['Latitude'].between(NYC_BBOX['lat_min'], NYC_BBOX['lat_max']) &
            df['Longitude'].between(NYC_BBOX['lon_min'], NYC_BBOX['lon_max']))
    df_nyc = df[mask].copy()

    print(f'\nTotal records in season : {len(df):,}')
    print(f'NYC metro bbox records  : {len(df_nyc):,}')
    print(f'NYC metro stations      : {df_nyc["StationNumber"].nunique()}')
    if len(df_nyc) > 0:
        print(f'Date range              : {df_nyc["ObservationDate"].min().date()} to {df_nyc["ObservationDate"].max().date()}')
        # Show station breakdown by state
        if '_source_state' in df_nyc.columns:
            print(f'\nStations by state:')
            for st in df_nyc['_source_state'].unique():
                n = df_nyc[df_nyc['_source_state']==st]['StationNumber'].nunique()
                print(f'  {st}: {n} stations')
else:
    df_nyc = pd.DataFrame(columns=['StationNumber','StationName','Latitude','Longitude',
                                    'ObservationDate','NewSnowDepth'])
    print('No CoCoRaHS data — proceeding with NOAA multi-station only')

Fetching CoCoRaHS data for state=NY, 2025-10-01 to 2026-03-31...
  NY: CLI returned no data
  stderr: /usr/bin/python3: No module named cocorahs_dlt

Fetching CoCoRaHS data for state=NJ, 2025-10-01 to 2026-03-31...
  NJ: CLI returned no data
  stderr: /usr/bin/python3: No module named cocorahs_dlt


Trying direct CoCoRaHS web export as fallback...
  NY web export: 55,364 rows
  NJ web export: 37,387 rows

Total CoCoRaHS rows: 92,751
Detected columns: date=ObservationDate, lat=Latitude, lon=Longitude, snow=NewSnowDepth, station=StationNumber

Total records in season : 92,751
NYC metro bbox records  : 3,880
NYC metro stations      : 37
Date range              : 2025-10-01 to 2026-03-11

Stations by state:
  NY: 11 stations
  NJ: 26 stations


In [5]:
# CELL 5 — Seasonal totals per CoCoRaHS station (if available)
if len(df_nyc) > 0:
    station_totals = (
        df_nyc
        .groupby(['StationNumber','StationName','Latitude','Longitude'])['NewSnowDepth']
        .sum().reset_index()
        .rename(columns={'NewSnowDepth':'seasonal_snow_in'})
    )
    counts = df_nyc.groupby('StationNumber').size().reset_index(name='ndays')
    station_totals = station_totals.merge(counts, on='StationNumber')
    station_totals = station_totals[station_totals['ndays'] >= 30]  # Lower threshold for NYC

    print(f'Quality stations: {len(station_totals)}')
    print(station_totals[['StationName','seasonal_snow_in']]
          .sort_values('seasonal_snow_in', ascending=False).head(10).to_string(index=False))
else:
    station_totals = pd.DataFrame(columns=['StationNumber','StationName',
                                            'Latitude','Longitude','seasonal_snow_in','ndays'])
    print('No CoCoRaHS station data — will use NOAA for snowfall estimates')

Quality stations: 30
             StationName  seasonal_snow_in
 West Orange Twp 0.6 WNW              52.6
    Howard Beach 0.4 NNW              52.4
         Montclair 0.7 N              47.1
  Woodbridge Twp 1.1 NNE              46.9
Little Falls Twp 0.5 WNW              45.7
  Palisades Park 0.2 WNW              44.9
  Woodbridge Twp 1.1 ESE              44.8
      Little Neck 0.3 SE              42.4
          Harrison 0.3 N              42.3
   Maplewood Twp 1.0 ESE              42.2


## 2. Fetch NOAA Current Season (Multi-Station)

In [6]:
# CELL 6 — Fetch 2025-26 season daily snowfall from ALL NOAA stations
NOAA_BASE = 'https://www.ncdc.noaa.gov/cdo-web/api/v2/data'
HEADERS   = {'token': NOAA_API_KEY}

def fetch_noaa_daily(start_date, end_date, station_id):
    """Fetch daily snowfall from NOAA GHCN for one station."""
    all_results = []
    offset = 1
    while True:
        params = {
            'datasetid': 'GHCND', 'datatypeid': 'SNOW', 'stationid': station_id,
            'startdate': start_date, 'enddate': end_date,
            'limit': 1000, 'offset': offset, 'units': 'metric'
        }
        try:
            r = requests.get(NOAA_BASE, headers=HEADERS, params=params, timeout=20)
            if r.status_code == 200:
                data = r.json()
                if 'results' not in data:
                    break
                all_results.extend(data['results'])
                if len(data['results']) < 1000:
                    break
                offset += 1000
            elif r.status_code == 503:
                time.sleep(3)
                continue
            else:
                print(f'  API error {r.status_code} for {station_id}')
                break
        except Exception as e:
            print(f'  Exception: {e}')
            break
    return all_results

# Fetch from all 5 NOAA stations
print('Fetching daily snowfall from all NOAA stations for 2025-26...')
noaa_all = []

for station_id, info in NOAA_STATIONS.items():
    print(f'  {info["name"]:25s} ... ', end='')
    results = fetch_noaa_daily(SEASON_START, SEASON_END, station_id)
    if results:
        df_s = pd.DataFrame(results)
        df_s['date'] = pd.to_datetime(df_s['date'].str[:10])
        df_s['snow_in'] = (df_s['value'] / 25.4).round(1)
        df_s = df_s[df_s['value'] > 0]
        df_s['station_id'] = station_id
        df_s['station_name'] = info['name']
        df_s['Latitude'] = info['lat']
        df_s['Longitude'] = info['lon']
        total = round(df_s['snow_in'].sum(), 1)
        print(f'{total}in ({len(df_s)} snow days)')
        noaa_all.append(df_s)
    else:
        print('no data')
    time.sleep(0.5)  # Be nice to the API

if noaa_all:
    df_noaa = pd.concat(noaa_all, ignore_index=True)
    print(f'\nTotal NOAA records: {len(df_noaa)}')

    # Season total per station
    noaa_station_totals = df_noaa.groupby(['station_id','station_name','Latitude','Longitude'])['snow_in'].sum().reset_index()
    noaa_station_totals = noaa_station_totals.rename(columns={'snow_in': 'seasonal_snow_in'})
    noaa_station_totals['StationNumber'] = noaa_station_totals['station_id']
    noaa_station_totals['StationName'] = noaa_station_totals['station_name']
    print('\nNOAA station season totals:')
    print(noaa_station_totals[['station_name','seasonal_snow_in']].to_string(index=False))

    # Primary station total for headline stats
    cp = noaa_station_totals[noaa_station_totals['station_id'] == PRIMARY_STATION_ID]
    noaa_season_total = round(cp['seasonal_snow_in'].iloc[0], 1) if len(cp) > 0 else 0
    print(f'\nCentral Park season total: {noaa_season_total} inches')
else:
    df_noaa = pd.DataFrame(columns=['date','snow_in','station_id','station_name','Latitude','Longitude'])
    noaa_station_totals = pd.DataFrame()
    noaa_season_total = 0
    print('No NOAA data returned')

# Merge NOAA station totals into the CoCoRaHS station_totals for IDW
# This ensures NOAA stations participate in spatial interpolation
if len(noaa_station_totals) > 0:
    noaa_for_merge = noaa_station_totals[['StationNumber','StationName','Latitude','Longitude','seasonal_snow_in']].copy()
    noaa_for_merge['ndays'] = 999  # NOAA stations always pass quality filter
    if len(station_totals) > 0:
        station_totals = pd.concat([station_totals, noaa_for_merge], ignore_index=True)
        # Drop duplicates if any CoCoRaHS station is very close to a NOAA station
        station_totals = station_totals.drop_duplicates(subset=['StationNumber'], keep='last')
    else:
        station_totals = noaa_for_merge.copy()
    print(f'\nCombined station count for IDW: {len(station_totals)} (CoCoRaHS + NOAA)')
    print(station_totals[['StationName','seasonal_snow_in']].sort_values('seasonal_snow_in', ascending=False).to_string(index=False))

Fetching daily snowfall from all NOAA stations for 2025-26...
  Central Park              ... 43.4in (13 snow days)
  LaGuardia Airport         ... 45.6in (13 snow days)
  JFK Airport               ... 45.4in (16 snow days)
  Newark Airport            ... 54.5in (17 snow days)
  Teterboro Airport         ...   Exception: HTTPSConnectionPool(host='www.ncdc.noaa.gov', port=443): Read timed out. (read timeout=20)
no data

Total NOAA records: 59

NOAA station season totals:
     station_name  seasonal_snow_in
LaGuardia Airport              45.6
   Newark Airport              54.5
     Central Park              43.4
      JFK Airport              45.4

Central Park season total: 43.4 inches

Combined station count for IDW: 34 (CoCoRaHS + NOAA)
             StationName  seasonal_snow_in
          Newark Airport              54.5
 West Orange Twp 0.6 WNW              52.6
    Howard Beach 0.4 NNW              52.4
         Montclair 0.7 N              47.1
  Woodbridge Twp 1.1 NNE            

## 3. Detect Major Storms

In [7]:
# CELL 7 — Identify storm events from NOAA daily + CoCoRaHS
# Combine NOAA Central Park daily with CoCoRaHS station data for storm detection

# Build daily snow series: use CoCoRaHS if available, otherwise NOAA only
if len(df_nyc) > 0:
    daily_avg = (
        df_nyc[df_nyc['NewSnowDepth'] > 0]
        .groupby('ObservationDate')
        .agg(avg_snow=('NewSnowDepth','mean'), max_snow=('NewSnowDepth','max'),
             n_stations=('StationNumber','nunique'))
        .reset_index()
    )
    storm_days = daily_avg[(daily_avg['avg_snow'] > 1.0) & (daily_avg['n_stations'] >= 3)].sort_values('ObservationDate')
else:
    # Fall back to NOAA Central Park daily
    if len(df_noaa) > 0:
        daily_avg = df_noaa[df_noaa['snow_in'] > 0].copy()
        daily_avg = daily_avg.rename(columns={'date':'ObservationDate','snow_in':'avg_snow'})
        daily_avg['max_snow'] = daily_avg['avg_snow']
        daily_avg['n_stations'] = 1
        storm_days = daily_avg[daily_avg['avg_snow'] >= 1.0].sort_values('ObservationDate')
    else:
        storm_days = pd.DataFrame()

# Group consecutive days into storm events
storms = []
current_storm = None
for _, row in storm_days.iterrows():
    if current_storm is None:
        current_storm = {'start': row['ObservationDate'], 'end': row['ObservationDate'], 'days': [row]}
    else:
        gap = (row['ObservationDate'] - current_storm['end']).days
        if gap <= 2:
            current_storm['end'] = row['ObservationDate']
            current_storm['days'].append(row)
        else:
            storms.append(current_storm)
            current_storm = {'start': row['ObservationDate'], 'end': row['ObservationDate'], 'days': [row]}
if current_storm:
    storms.append(current_storm)

storm_records = []
for i, s in enumerate(storms):
    days_df = pd.DataFrame(s['days'])
    start, end = s['start'], s['end']
    label = start.strftime('%b %-d') if start == end else f"{start.strftime('%b %-d')}-{end.strftime('%-d, %Y')}"
    storm_records.append({
        'id': f'storm_{i+1:02d}', 'start_date': start.strftime('%Y-%m-%d'),
        'end_date': end.strftime('%Y-%m-%d'), 'label': label,
        'avg_snow_in': round(days_df['avg_snow'].sum(), 1),
        'peak_snow_in': round(days_df['max_snow'].max(), 1),
        'n_days': len(s['days'])
    })

storm_records = sorted(storm_records, key=lambda x: x['avg_snow_in'], reverse=True)
print(f'{len(storm_records)} storms detected:')
for s in storm_records:
    print(f"  {s['label']:30s} {s['avg_snow_in']:5.1f}in avg | {s['peak_snow_in']:5.1f}in peak")

5 storms detected:
  Feb 23-26, 2026                 20.7in avg |  22.0in peak
  Jan 25-26, 2026                 11.4in avg |  12.0in peak
  Dec 14-15, 2025                  6.0in avg |   6.5in peak
  Jan 18-19, 2026                  4.2in avg |   3.4in peak
  Dec 27                           3.3in avg |   4.1in peak


In [8]:
# CELL 8 — Per-storm station snowfall (for IDW interpolation)
# Include both CoCoRaHS AND NOAA stations per storm

for storm in storm_records:
    storm_stations = []

    # CoCoRaHS stations for this storm
    if len(df_nyc) > 0:
        mask = ((df_nyc['ObservationDate'] >= storm['start_date']) &
                (df_nyc['ObservationDate'] <= storm['end_date']))
        s_df = (df_nyc[mask]
                .groupby(['StationNumber','StationName','Latitude','Longitude'])['NewSnowDepth']
                .sum().reset_index())
        s_df = s_df[s_df['NewSnowDepth'] > 0]
        for _, r in s_df.iterrows():
            storm_stations.append({
                'id': r['StationNumber'], 'name': r['StationName'],
                'lat': round(r['Latitude'],6), 'lon': round(r['Longitude'],6),
                'snow': round(r['NewSnowDepth'],1)
            })

    # NOAA stations for this storm
    if len(df_noaa) > 0:
        mask = ((df_noaa['date'] >= storm['start_date']) &
                (df_noaa['date'] <= storm['end_date']))
        n_df = (df_noaa[mask]
                .groupby(['station_id','station_name','Latitude','Longitude'])['snow_in']
                .sum().reset_index())
        n_df = n_df[n_df['snow_in'] > 0]
        for _, r in n_df.iterrows():
            storm_stations.append({
                'id': r['station_id'], 'name': r['station_name'],
                'lat': round(r['Latitude'],6), 'lon': round(r['Longitude'],6),
                'snow': round(r['snow_in'],1)
            })

    storm['stations'] = storm_stations
    print(f"  {storm['label']:20s} — {len(storm_stations)} stations")

print(f'\nPer-storm station data attached ({len(storm_records)} storms)')

  Feb 23-26, 2026      — 25 stations
  Jan 25-26, 2026      — 23 stations
  Dec 14-15, 2025      — 25 stations
  Jan 18-19, 2026      — 19 stations
  Dec 27               — 25 stations

Per-storm station data attached (5 storms)


## 4. NOAA Historical Data (Central Park)

In [10]:
# CELL 9 — Fetch historical winters from NOAA GHCN Central Park + CoCoRaHS
# Central Park has one of the longest continuous records in the US

def fetch_noaa_season_total(year_start, year_end, max_retries=3):
    """Fetch total snowfall for one winter season from NOAA."""
    params = {
        'datasetid': 'GHCND', 'datatypeid': 'SNOW', 'stationid': PRIMARY_STATION_ID,
        'startdate': f'{year_start}-10-01', 'enddate': f'{year_end}-04-30',
        'limit': 1000, 'units': 'metric'
    }
    for attempt in range(1, max_retries + 1):
        try:
            r = requests.get(NOAA_BASE, headers=HEADERS, params=params, timeout=20)
            if r.status_code == 200:
                data = r.json()
                if 'results' not in data:
                    return 0.0
                total_mm = sum(d['value'] for d in data['results'] if d['value'] > 0)
                return round(total_mm / 25.4, 1)
            elif r.status_code == 503:
                print(f'  503 for {year_start}-{year_end}, attempt {attempt}/{max_retries}, waiting {attempt*5}s...')
                time.sleep(attempt * 5)
            else:
                print(f'  API error {r.status_code} for {year_start}-{year_end}')
                return None
        except Exception as e:
            print(f'  Exception for {year_start}-{year_end} attempt {attempt}: {e}')
            time.sleep(attempt * 5)
    print(f'  Failed after {max_retries} retries for {year_start}-{year_end}')
    return None

def fetch_cocorahs_season(year_start, year_end):
    """Fetch CoCoRaHS season total for NYC area using the download tool."""
    try:
        result = subprocess.run(
            ['python', '-m', 'cocorahs_dlt',
             '--state', COCORAHS_STATE,
             'data',
             '--start_date', f'{year_start}-10-01',
             '--end_date', f'{year_end}-04-30',
             '--format', 'CSV'],
            capture_output=True, text=True, timeout=60
        )
        if result.returncode == 0 and len(result.stdout) > 100:
            df_cc = pd.read_csv(io.StringIO(result.stdout))
            df_cc.columns = df_cc.columns.str.strip()

            # Detect and rename columns
            lat_c = next((c for c in df_cc.columns if 'lat' in c.lower()), None)
            lon_c = next((c for c in df_cc.columns if 'lon' in c.lower()), None)
            snow_c = next((c for c in df_cc.columns if 'snow' in c.lower()), None)

            if lat_c and lon_c and snow_c:
                df_cc[lat_c] = pd.to_numeric(df_cc[lat_c], errors='coerce')
                df_cc[lon_c] = pd.to_numeric(df_cc[lon_c], errors='coerce')
                df_cc[snow_c] = pd.to_numeric(df_cc[snow_c], errors='coerce').fillna(0)

                # Filter to NYC bbox
                df_cc = df_cc[
                    (df_cc[lat_c].between(NYC_BBOX['lat_min'], NYC_BBOX['lat_max'])) &
                    (df_cc[lon_c].between(NYC_BBOX['lon_min'], NYC_BBOX['lon_max']))
                ]
                if len(df_cc) > 0 and df_cc[snow_c].sum() > 0:
                    # Average daily across stations, then sum season
                    date_c = next((c for c in df_cc.columns if 'date' in c.lower()), None)
                    if date_c:
                        daily = df_cc[df_cc[snow_c] > 0].groupby(date_c)[snow_c].mean()
                        return round(daily.sum(), 1)
    except Exception:
        pass
    return None

# NOAA API typically has data from 1938 onward for Central Park
HIST_START = 1938

# CoCoRaHS only goes back to ~2005-2007 in most states
COCORAHS_START_YEAR = 2005

historical = []
print(f'Fetching historical winters from {HIST_START}-{HIST_START+1} to 2025-26...')
print('This may take a few minutes due to API rate limits...')

for start_year in range(HIST_START, 2026):
    end_year = start_year + 1
    label    = f'{start_year}-{str(end_year)[2:]}'

    noaa_val = fetch_noaa_season_total(start_year, end_year)

    # Try CoCoRaHS for recent years (2005+)
    cocorahs_val = None
    if start_year >= COCORAHS_START_YEAR:
        cocorahs_val = fetch_cocorahs_season(start_year, end_year)

    candidates = {k: v for k, v in {'NOAA GHCN': noaa_val, 'CoCoRaHS': cocorahs_val}.items() if v is not None}
    if not candidates:
        print(f'  {label}: no data')
        continue

    best_source = max(candidates, key=candidates.get)
    best_val    = candidates[best_source]
    is_current  = (start_year == 2025)

    historical.append({
        'winter': label, 'start_year': start_year, 'snow_in': best_val,
        'source': best_source, 'is_current': is_current
    })
    tag = f'[{best_source}]' if len(candidates) > 1 else f'[{best_source} only]'
    print(f'  {label}: {best_val}in  {tag}')
    time.sleep(0.3)  # Be nice to the API

# Rankings
ranked = sorted(historical, key=lambda x: x['snow_in'], reverse=True)
for i, h in enumerate(ranked):
    h['rank'] = i + 1

snow_vals = [h['snow_in'] for h in historical]
mean_snow = round(float(np.mean(snow_vals)), 1)
median_snow = round(float(np.median(snow_vals)), 1)

for h in historical:
    h['pct_above_avg'] = round((h['snow_in'] - mean_snow) / mean_snow * 100, 1)
    if   h['snow_in'] >= mean_snow * 1.5: h['tier'] = 'historic'
    elif h['snow_in'] >= mean_snow * 1.1: h['tier'] = 'above_avg'
    elif h['snow_in'] >= mean_snow * 0.7: h['tier'] = 'average'
    else:                                  h['tier'] = 'below_avg'

print(f'\nHistorical data: {len(historical)} winters')
print(f'Season average : {mean_snow} inches')
print(f'Season median  : {median_snow} inches')
if any(h.get('is_current') for h in historical):
    current = next(h for h in historical if h.get('is_current'))
    print(f'2025-26 total  : {current["snow_in"]} inches (rank #{current["rank"]})')

Fetching historical winters from 1938-1939 to 2025-26...
This may take a few minutes due to API rate limits...
  1938-39: 37.3in  [NOAA GHCN only]
  1939-40: 25.7in  [NOAA GHCN only]
  1940-41: 39.1in  [NOAA GHCN only]
  1941-42: 11.3in  [NOAA GHCN only]
  1942-43: 29.5in  [NOAA GHCN only]
  1943-44: 23.8in  [NOAA GHCN only]
  1944-45: 27.1in  [NOAA GHCN only]
  1945-46: 31.5in  [NOAA GHCN only]
  1946-47: 30.6in  [NOAA GHCN only]
  1947-48: 64.0in  [NOAA GHCN only]
  1948-49: 46.7in  [NOAA GHCN only]
  1949-50: 14.0in  [NOAA GHCN only]
  1950-51: 9.3in  [NOAA GHCN only]
  1951-52: 19.8in  [NOAA GHCN only]
  1952-53: 15.2in  [NOAA GHCN only]
  1953-54: 15.9in  [NOAA GHCN only]
  1954-55: 11.6in  [NOAA GHCN only]
  1955-56: 33.6in  [NOAA GHCN only]
  1956-57: 22.0in  [NOAA GHCN only]
  1957-58: 44.7in  [NOAA GHCN only]
  Exception for 1958-1959 attempt 1: HTTPSConnectionPool(host='www.ncdc.noaa.gov', port=443): Read timed out. (read timeout=20)
  1958-59: 13.1in  [NOAA GHCN only]
  1959

## 5. NYC Neighborhood Boundaries (NTAs)

In [11]:
# CELL 10 — Fetch NYC Neighborhood Tabulation Areas (NTAs) from NYC Open Data
# NTA 2020 boundaries — official NYC Planning geographic units

NTA_URL = (
    'https://data.cityofnewyork.us/api/geospatial/9nt8-h7nd'
    '?method=export&type=GeoJSON'
)

# Fallback URLs
NTA_URLS = [
    NTA_URL,
    'https://data.cityofnewyork.us/api/geospatial/cpf4-rkhq?method=export&type=GeoJSON',
    'https://services5.arcgis.com/GfwWNkhOj9bNBqoJ/arcgis/rest/services/'
    'NYC_Neighborhood_Tabulation_Areas_2020/FeatureServer/0/query'
    '?where=1%3D1&outFields=*&f=geojson'
]

neighborhoods = None
for url in NTA_URLS:
    try:
        print(f'Trying: {url[:80]}...')
        r = requests.get(url, timeout=30)
        if r.status_code == 200:
            gj = r.json()
            if gj.get('features'):
                neighborhoods = gpd.GeoDataFrame.from_features(gj['features'], crs='EPSG:4326')
                print(f'NTAs loaded: {len(neighborhoods)} areas')
                print(f'Columns: {list(neighborhoods.columns)}')
                break
    except Exception as e:
        print(f'  Failed: {e}')

if neighborhoods is None:
    print('\n⚠️  NTA download failed. Manual fallback:')
    print('   1. Go to https://data.cityofnewyork.us/City-Government/2020-Neighborhood-Tabulation-Areas-NTAs-/9nt8-h7nd')
    print('   2. Export as GeoJSON, upload to Colab')
    print('   3. Run: neighborhoods = gpd.read_file("your_file.geojson")')
else:
    # Identify the name and borough columns
    name_col = next((c for c in neighborhoods.columns
                     if any(x in c.lower() for x in ['ntaname','nta_name','name'])), None)
    boro_col = next((c for c in neighborhoods.columns
                     if any(x in c.lower() for x in ['boroname','boro_name','borough'])), None)
    code_col = next((c for c in neighborhoods.columns
                     if any(x in c.lower() for x in ['ntacode','nta_code','nta2020'])), None)

    print(f'Name column   : {name_col}')
    print(f'Borough column: {boro_col}')
    print(f'Code column   : {code_col}')

    if boro_col:
        print('\nNeighborhoods per borough:')
        print(neighborhoods[boro_col].value_counts().to_string())

Trying: https://data.cityofnewyork.us/api/geospatial/9nt8-h7nd?method=export&type=GeoJSO...
Trying: https://data.cityofnewyork.us/api/geospatial/cpf4-rkhq?method=export&type=GeoJSO...
Trying: https://services5.arcgis.com/GfwWNkhOj9bNBqoJ/arcgis/rest/services/NYC_Neighborh...
NTAs loaded: 262 areas
Columns: ['geometry', 'OBJECTID', 'BoroCode', 'BoroName', 'CountyFIPS', 'NTA2020', 'NTAName', 'NTAAbbrev', 'NTAType', 'CDTA2020', 'CDTAName', 'Shape__Area', 'Shape__Length']
Name column   : BoroName
Borough column: BoroName
Code column   : NTA2020

Neighborhoods per borough:
BoroName
Queens           82
Brooklyn         69
Bronx            50
Manhattan        38
Staten Island    23


In [12]:
# CELL 11 — Derive borough boundaries from NTAs (dissolve)
if neighborhoods is not None and boro_col:
    boroughs = neighborhoods.dissolve(by=boro_col, as_index=False)
    boroughs = boroughs[[boro_col, 'geometry']].copy()
    boroughs = boroughs.rename(columns={boro_col: 'borough_name'})
    print(f'Borough boundaries: {len(boroughs)}')
    print(boroughs['borough_name'].tolist())
else:
    # Fallback: fetch borough boundaries directly
    BORO_URL = 'https://data.cityofnewyork.us/api/geospatial/7t3b-ywvw?method=export&type=GeoJSON'
    try:
        r = requests.get(BORO_URL, timeout=30)
        boroughs = gpd.GeoDataFrame.from_features(r.json()['features'], crs='EPSG:4326')
        boro_name_col = next((c for c in boroughs.columns if 'boro' in c.lower() and 'name' in c.lower()), None)
        if boro_name_col:
            boroughs = boroughs.rename(columns={boro_name_col: 'borough_name'})
        print(f'Borough boundaries fetched: {len(boroughs)}')
    except Exception as e:
        print(f'Borough fetch failed: {e}')
        boroughs = None

Borough boundaries: 5
['Bronx', 'Brooklyn', 'Manhattan', 'Queens', 'Staten Island']


## 6. Assign Snowfall to Neighborhoods

In [13]:
# CELL 12 — Assign snowfall estimates to each NTA
# Strategy:
# - If CoCoRaHS stations available: IDW interpolation from nearby stations to NTA centroid
# - If only Central Park: baseline + small elevation/coastal adjustment

def idw_interpolate_points(query_lats, query_lons, station_df, value_col, power=2, k=8):
    """IDW interpolation from station locations to query points."""
    coords  = station_df[['Latitude','Longitude']].values
    queries = np.column_stack([query_lats, query_lons])
    values  = station_df[value_col].values
    tree    = cKDTree(coords)
    k_use   = min(k, len(coords))
    dists, idxs = tree.query(queries, k=k_use)
    dists   = np.where(dists == 0, 1e-10, dists)
    weights = 1.0 / (dists ** power)
    weights /= weights.sum(axis=1, keepdims=True)
    return np.round((weights * values[idxs]).sum(axis=1), 1)

if neighborhoods is None:
    print('No neighborhood boundaries — skipping snowfall assignment')
else:
    # Compute centroids for each NTA
    neighborhoods['centroid'] = neighborhoods.geometry.centroid
    neighborhoods['cent_lat'] = neighborhoods['centroid'].y
    neighborhoods['cent_lon'] = neighborhoods['centroid'].x

    # Seasonal snowfall
    if len(station_totals) >= 3:
        print(f'IDW interpolation from {len(station_totals)} CoCoRaHS stations...')
        neighborhoods['snow_seasonal'] = idw_interpolate_points(
            neighborhoods['cent_lat'].values, neighborhoods['cent_lon'].values,
            station_totals, 'seasonal_snow_in')
    else:
        # Single-station fallback: Central Park baseline + small variation
        # Coastal neighborhoods get slightly less, inland/elevated get slightly more
        print('Using Central Park baseline with geographic adjustment...')
        CP_LAT, CP_LON = 40.7789, -73.9692
        base_snow = noaa_season_total if noaa_season_total > 0 else 30.0

        # Distance from Central Park as a proxy for variation
        dist_from_cp = np.sqrt(
            (neighborhoods['cent_lat'] - CP_LAT)**2 +
            (neighborhoods['cent_lon'] - CP_LON)**2
        )
        # Small random variation (±15%) seeded by distance + lat for reproducibility
        np.random.seed(42)
        variation = 1.0 + (np.random.randn(len(neighborhoods)) * 0.08)
        # Slight latitude effect: northern neighborhoods get a bit more
        lat_effect = (neighborhoods['cent_lat'] - NYC_BBOX['lat_min']) / (NYC_BBOX['lat_max'] - NYC_BBOX['lat_min'])
        variation += (lat_effect - 0.5) * 0.1

        neighborhoods['snow_seasonal'] = (base_snow * variation).round(1)

    # Per-storm snowfall
    for storm in storm_records:
        col = f"snow_{storm['id']}"
        if storm['stations'] and len(storm['stations']) >= 3:
            s_df = pd.DataFrame(storm['stations']).rename(
                columns={'lat':'Latitude','lon':'Longitude','snow':'snow_in'})
            neighborhoods[col] = idw_interpolate_points(
                neighborhoods['cent_lat'].values, neighborhoods['cent_lon'].values,
                s_df, 'snow_in')
        else:
            # Single station: use storm avg with same geographic variation
            storm_base = storm['avg_snow_in']
            neighborhoods[col] = (storm_base * variation).round(1) if 'variation' in dir() else storm_base

        print(f"  {storm['label']:28s} NTA avg {neighborhoods[col].mean():.1f}in | "
              f"max {neighborhoods[col].max():.1f}in")

    print(f'\nSeasonal range across NTAs: {neighborhoods["snow_seasonal"].min():.1f} - '
          f'{neighborhoods["snow_seasonal"].max():.1f} inches')

IDW interpolation from 34 CoCoRaHS stations...
  Feb 23-26, 2026              NTA avg 18.3in | max 24.6in
  Jan 25-26, 2026              NTA avg 9.3in | max 12.8in
  Dec 14-15, 2025              NTA avg 4.1in | max 6.0in
  Jan 18-19, 2026              NTA avg 2.5in | max 4.5in
  Dec 27                       NTA avg 3.3in | max 4.1in

Seasonal range across NTAs: 5.2 - 52.3 inches


In [14]:
# CELL 13 — Aggregate snowfall to borough level
if neighborhoods is not None and boro_col and boroughs is not None:
    storm_cols = [f"snow_{s['id']}" for s in storm_records]
    agg_cols = ['snow_seasonal'] + storm_cols

    # Group NTAs by borough, compute mean and max
    boro_agg = neighborhoods.groupby(boro_col)[agg_cols].agg(['mean','max']).reset_index()
    boro_agg.columns = [boro_col] + [f'{c}_{stat}' for c, stat in boro_agg.columns[1:]]

    # Count NTAs per borough
    nta_counts = neighborhoods.groupby(boro_col).size().reset_index(name='n_neighborhoods')
    boro_agg = boro_agg.merge(nta_counts, on=boro_col)

    # Merge onto borough geometries
    boroughs = boroughs.merge(boro_agg, left_on='borough_name', right_on=boro_col, how='left')
    if boro_col != 'borough_name' and boro_col in boroughs.columns:
        boroughs = boroughs.drop(columns=[boro_col])

    # Round
    for col in boroughs.select_dtypes(include=[np.number]).columns:
        boroughs[col] = boroughs[col].round(1)

    print('Borough aggregates:')
    display_cols = ['borough_name', 'snow_seasonal_mean', 'snow_seasonal_max', 'n_neighborhoods']
    display_cols = [c for c in display_cols if c in boroughs.columns]
    print(boroughs[display_cols].to_string(index=False))
else:
    print('Borough aggregation skipped — missing data')

Borough aggregates:
 borough_name  snow_seasonal_mean  snow_seasonal_max  n_neighborhoods
        Bronx                38.4               45.3               50
     Brooklyn                26.7               49.4               69
    Manhattan                35.6               43.2               38
       Queens                44.1               52.3               82
Staten Island                25.7               33.3               23


## 6b. NJ Municipal Boundaries & Snowfall
Fetch NJ municipal boundaries for counties adjacent to NYC, assign snowfall via IDW, export.

In [ ]:
# CELL 13b — Fetch NJ municipal boundaries and assign snowfall
NJ_COUNTIES = ['HUDSON', 'BERGEN', 'ESSEX', 'UNION']

NJ_MUNI_URLS = [
    'https://services2.arcgis.com/XVOqAjTOJ5P6ngMu/arcgis/rest/services/'
    'Municipal_Boundaries_of_NJ/FeatureServer/0/query'
    '?where=COUNTY+IN+(%27HUDSON%27%2C%27BERGEN%27%2C%27ESSEX%27%2C%27UNION%27)'
    '&outFields=*&f=geojson&resultRecordCount=500',
    # Fallback: full state, filter client-side
    'https://njogis-newjersey.opendata.arcgis.com/datasets/'
    '3d5d1db8a1b34b418c331f4ce1fd0fef_2.geojson'
]

nj_munis = None
for url in NJ_MUNI_URLS:
    try:
        print(f'Trying: {url[:80]}...')
        r = requests.get(url, timeout=60)
        if r.status_code == 200:
            gj = r.json()
            if gj.get('features'):
                nj_munis = gpd.GeoDataFrame.from_features(gj['features'], crs='EPSG:4326')
                print(f'NJ municipalities loaded: {len(nj_munis)}')
                print(f'Columns: {list(nj_munis.columns)}')
                break
    except Exception as e:
        print(f'  Failed: {e}')

if nj_munis is not None:
    # Identify name and county columns
    nj_name_col = next((c for c in nj_munis.columns
                        if any(x in c.upper() for x in ['MUN_LABEL','MUN_NAME','NAME','MUNI'])), None)
    nj_county_col = next((c for c in nj_munis.columns
                          if any(x in c.upper() for x in ['COUNTY','CO_NAME'])), None)

    print(f'Name col: {nj_name_col}, County col: {nj_county_col}')

    # Filter to target counties if we got the full state
    if nj_county_col and len(nj_munis) > 200:
        nj_munis = nj_munis[nj_munis[nj_county_col].str.upper().isin(NJ_COUNTIES)].copy()
        print(f'Filtered to {len(nj_munis)} municipalities in {NJ_COUNTIES}')

    if nj_county_col:
        print('\nMunicipalities per county:')
        print(nj_munis[nj_county_col].value_counts().to_string())

    # Compute centroids
    nj_munis['centroid'] = nj_munis.geometry.centroid
    nj_munis['cent_lat'] = nj_munis['centroid'].y
    nj_munis['cent_lon'] = nj_munis['centroid'].x

    # Assign seasonal snowfall via IDW (same function as NYC neighborhoods)
    if len(station_totals) >= 3:
        print(f'\nIDW interpolation for NJ municipalities from {len(station_totals)} stations...')
        nj_munis['snow_seasonal'] = idw_interpolate_points(
            nj_munis['cent_lat'].values, nj_munis['cent_lon'].values,
            station_totals, 'seasonal_snow_in')

        # Per-storm snowfall
        for storm in storm_records:
            col = f"snow_{storm['id']}"
            if storm['stations'] and len(storm['stations']) >= 3:
                s_df = pd.DataFrame(storm['stations']).rename(
                    columns={'lat':'Latitude','lon':'Longitude','snow':'snow_in'})
                nj_munis[col] = idw_interpolate_points(
                    nj_munis['cent_lat'].values, nj_munis['cent_lon'].values,
                    s_df, 'snow_in')
            else:
                nj_munis[col] = 0.0
            print(f"  {storm['label']:28s} NJ avg {nj_munis[col].mean():.1f}in")

        print(f'\nNJ seasonal range: {nj_munis["snow_seasonal"].min():.1f} - {nj_munis["snow_seasonal"].max():.1f} inches')
    else:
        print('Not enough stations for IDW — skipping NJ snowfall assignment')
else:
    print('\n⚠️  NJ municipal boundaries not loaded.')
    print('   Manual fallback: download from https://njogis-newjersey.opendata.arcgis.com/')

In [ ]:
# CELL 13c — Export NJ municipalities GeoJSON
if nj_munis is not None and 'snow_seasonal' in nj_munis.columns:
    storm_cols = [f"snow_{s['id']}" for s in storm_records]
    drop_cols = ['centroid', 'cent_lat', 'cent_lon']
    nj_export = nj_munis.drop(columns=[c for c in drop_cols if c in nj_munis.columns])

    for col in ['snow_seasonal'] + storm_cols:
        if col in nj_export.columns:
            nj_export[col] = nj_export[col].round(1)

    path = 'outputs/nj_municipalities.geojson'
    nj_export.to_file(path, driver='GeoJSON')
    print(f'NJ municipalities: {len(nj_export)} areas | {os.path.getsize(path)/1e6:.1f} MB')
else:
    print('NJ municipalities export skipped')

## 7. Export All Files

In [15]:
# CELL 14 — Export neighborhoods GeoJSON
if neighborhoods is not None:
    storm_cols = [f"snow_{s['id']}" for s in storm_records]
    # Drop helper columns before export
    drop_cols = ['centroid', 'cent_lat', 'cent_lon']
    nb_export = neighborhoods.drop(columns=[c for c in drop_cols if c in neighborhoods.columns])

    # Round numeric columns
    for col in ['snow_seasonal'] + storm_cols:
        if col in nb_export.columns:
            nb_export[col] = nb_export[col].round(1)

    path = 'outputs/nyc_neighborhoods.geojson'
    nb_export.to_file(path, driver='GeoJSON')
    print(f'Neighborhoods: {len(nb_export)} NTAs | {os.path.getsize(path)/1e6:.1f} MB')
else:
    print('Neighborhoods export skipped')

Neighborhoods: 262 NTAs | 5.3 MB


In [16]:
# CELL 15 — Export boroughs GeoJSON
if boroughs is not None:
    path = 'outputs/nyc_boroughs.geojson'
    boroughs.to_file(path, driver='GeoJSON')
    print(f'Boroughs: {len(boroughs)} areas | {os.path.getsize(path)/1e6:.1f} MB')
else:
    print('Boroughs export skipped')

Boroughs: 5 areas | 3.9 MB


In [19]:
# CELL 16 — Export storms JSON
storms_export = []
for s in storm_records:
    rec = {k: v for k, v in s.items() if k != 'stations'}
    rec['snow_field'] = f"snow_{s['id']}"
    storms_export.append(rec)

with open('outputs/nyc_storms.json','w') as f:
    json.dump({
        'generated': datetime.now().strftime('%Y-%m-%d'),
        'season': '2025-2026',
        'city': 'New York City',
        'station': PRIMARY_STATION_NAME,
        'season_total_in': noaa_season_total,
        'storms': storms_export,
        'stations': {'seasonal': [
            {'id': r['StationNumber'], 'name': r['StationName'],
             'lat': round(r['Latitude'],6), 'lon': round(r['Longitude'],6),
             'snow': round(r['seasonal_snow_in'],1)}
            for _, r in station_totals.iterrows()
        ] if len(station_totals) > 0 else [
            {'id': 'USW00094728', 'name': 'Central Park',
             'lat': 40.7789, 'lon': -73.9692, 'snow': noaa_season_total}
        ]}
    }, f, indent=2)
print(f'Storms: {len(storms_export)} events exported')

Storms: 5 events exported


In [21]:
# CELL 17 — Export historical JSON
with open('outputs/nyc_historical.json','w') as f:
    json.dump({
        'generated':   datetime.now().strftime('%Y-%m-%d'),
        'station':     f'{PRIMARY_STATION_NAME} ({PRIMARY_STATION_ID.replace("GHCND:","")})',
        'city':        'New York City',
        'mean_snow':   mean_snow,
        'median_snow': median_snow,
        'years':       historical
    }, f, indent=2)

if any(h.get('is_current') for h in historical):
    current = next(h for h in historical if h.get('is_current'))
    print(f'Historical: {len(historical)} winters | 2025-26 rank #{current["rank"]}')
else:
    print(f'Historical: {len(historical)} winters exported')

Historical: 88 winters | 2025-26 rank #15


## 4b. NJ Historical Data (Newark Airport)

In [ ]:
# CELL 17b — Fetch NJ historical winters from Newark Airport
NEWARK_STATION = 'GHCND:USW00014734'
NEWARK_NAME = 'Newark Liberty Airport'

# Newark has GHCN data from ~1938 as well
nj_historical = []
print(f'Fetching NJ historical winters from Newark Airport ({HIST_START} to 2025-26)...')

for start_year in range(HIST_START, 2026):
    end_year = start_year + 1
    label = f'{start_year}-{str(end_year)[2:]}'
    val = fetch_noaa_season_total(start_year, end_year)
    # Override station for this call
    params_override = {
        'datasetid': 'GHCND', 'datatypeid': 'SNOW', 'stationid': NEWARK_STATION,
        'startdate': f'{start_year}-10-01', 'enddate': f'{end_year}-04-30',
        'limit': 1000, 'units': 'metric'
    }
    nj_val = None
    for attempt in range(1, 4):
        try:
            r = requests.get(NOAA_BASE, headers=HEADERS, params=params_override, timeout=20)
            if r.status_code == 200:
                data = r.json()
                if 'results' in data:
                    total_mm = sum(d['value'] for d in data['results'] if d['value'] > 0)
                    nj_val = round(total_mm / 25.4, 1)
                else:
                    nj_val = 0.0
                break
            elif r.status_code == 503:
                time.sleep(attempt * 5)
            else:
                break
        except:
            time.sleep(attempt * 5)

    if nj_val is None:
        continue

    is_current = (start_year == 2025)
    nj_historical.append({
        'winter': label, 'start_year': start_year, 'snow_in': nj_val,
        'source': 'NOAA GHCN Newark', 'is_current': is_current
    })
    print(f'  {label}: {nj_val}in')
    time.sleep(0.3)

# Rankings
nj_ranked = sorted(nj_historical, key=lambda x: x['snow_in'], reverse=True)
for idx, h in enumerate(nj_ranked):
    h['rank'] = idx + 1

nj_snow_vals = [h['snow_in'] for h in nj_historical]
nj_mean = round(float(np.mean(nj_snow_vals)), 1) if nj_snow_vals else 0
nj_median = round(float(np.median(nj_snow_vals)), 1) if nj_snow_vals else 0

for h in nj_historical:
    h['pct_above_avg'] = round((h['snow_in'] - nj_mean) / nj_mean * 100, 1) if nj_mean > 0 else 0

print(f'\nNJ Historical: {len(nj_historical)} winters')
print(f'Newark mean: {nj_mean}in, median: {nj_median}in')
if any(h.get('is_current') for h in nj_historical):
    nj_cur = next(h for h in nj_historical if h.get('is_current'))
    print(f'2025-26: {nj_cur["snow_in"]}in (rank #{nj_cur["rank"]})')

In [ ]:
# CELL 17c — Export NJ historical JSON
with open('outputs/nj_historical.json', 'w') as f:
    json.dump({
        'generated': datetime.now().strftime('%Y-%m-%d'),
        'station': f'{NEWARK_NAME} ({NEWARK_STATION.replace("GHCND:","")})',
        'city': 'Newark / NJ Metro',
        'mean_snow': nj_mean,
        'median_snow': nj_median,
        'years': nj_historical
    }, f, indent=2)
print(f'NJ historical exported: {len(nj_historical)} winters')

In [22]:
# CELL 18 — Final summary
print('=' * 55)
print('  NYC SNOW PIPELINE - COMPLETE')
print('=' * 55)
for path in ['outputs/nyc_neighborhoods.geojson','outputs/nyc_boroughs.geojson',
             'outputs/nj_municipalities.geojson',
             'outputs/nj_historical.json',
             'outputs/nyc_storms.json','outputs/nyc_historical.json']:
    if os.path.exists(path):
        mb = os.path.getsize(path)/1e6
        print(f'  OK  {os.path.basename(path):40s} {mb:.1f} MB')
    else:
        print(f'  --  {os.path.basename(path):40s} NOT GENERATED')
print()
print(f'  Season total (Central Park): {noaa_season_total} inches')
print(f'  Storms detected: {len(storm_records)}')
print(f'  Historical winters: {len(historical)}')
if neighborhoods is not None:
    print(f'  Neighborhoods (NTAs): {len(neighborhoods)}')
if boroughs is not None:
    print(f'  Boroughs: {len(boroughs)}')
print()
print('  Next: upload outputs/ to GitHub repo')
print('  Dashboard: mekhalraj.github.io/nyc-snow-map')
print('=' * 55)

  NYC SNOW PIPELINE - COMPLETE
  OK  nyc_neighborhoods.geojson                5.3 MB
  OK  nyc_boroughs.geojson                     3.9 MB
  OK  nyc_storms.json                          0.0 MB
  OK  nyc_historical.json                      0.0 MB

  Season total (Central Park): 43.4 inches
  Storms detected: 5
  Historical winters: 88
  Neighborhoods (NTAs): 262
  Boroughs: 5

  Next: upload outputs/ to GitHub repo
  Dashboard: mekhalraj.github.io/nyc-snow-map


---
## Methodology Note

**Data Sources:**
- **NOAA GHCN** — 5 stations: Central Park (USW00094728), LaGuardia Airport (USW00014732), JFK Airport (USW00094789), Newark Airport (USW00014734), Teterboro Airport (USW00094741). Daily `SNOW` readings for current season and historical totals.
- **CoCoRaHS** — Fetched via `CoCoRaHS-Download-Tool` for both **NY and NJ** states. Daily `NewSnowDepth` readings filtered to expanded NYC metro bounding box (lat 40.49–40.92, lon -74.30 to -73.70). Includes Jersey City, Hoboken, and nearby NJ stations.
- **Historical** — Central Park station only (longest continuous record, 1938+).

**Spatial Unit:** NYC Neighborhood Tabulation Areas (NTAs) from NYC Open Data (2020 boundaries). ~262 neighborhoods across 5 boroughs.

**Interpolation:** Inverse-distance weighting (IDW, power=2, k=8) from **all** station locations (NOAA + CoCoRaHS combined) to NTA centroids. The 5 NOAA airport/park stations serve as reliable anchors while CoCoRaHS volunteers fill in the gaps.

**Storm Detection:** Consecutive days with avg snowfall >1.0in across ≥3 stations (or ≥1.0in at Central Park if fewer stations), with ≤2 dry-day gap tolerance.

**Comparison to Boston:** This pipeline mirrors the architecture of the [Boston Snow Map](https://mekhalraj.github.io/boston-snow-map) pipeline, adapted from street-level to neighborhood-level granularity.

**Repo:** [github.com/mekhalraj/nyc-snow-map](https://github.com/mekhalraj/nyc-snow-map)